# Table-CNN MRC Colab runner
Use an H100 or another high-memory NVIDIA runtime. The repository URL is preconfigured. Store a Hugging Face token under the `HF_TOKEN` key in Colab Secrets if your environment requires one.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    !git -C {REPO_DIR} pull --ff-only
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

token = userdata.get("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token

## Verify the full model path

In [ ]:
!python scripts/smoke_test.py --config configs/baseline.yaml

## Train (run after the smoke test passes)

In [ ]:
DRIVE_OUTPUT = "/content/drive/MyDrive/cnn_qwen_table_mcr/outputs/baseline"
!python scripts/run_experiment.py --config configs/baseline.yaml --mirror-output-dir {DRIVE_OUTPUT}

## Optional: run a disconnect-safe ordered sweep
Rerun this cell after a disconnect. Completed configs are skipped and the interrupted config resumes from its latest Drive checkpoint. Edit the ordered list to choose experiments.

In [ ]:
SWEEP_CONFIGS = [
    "configs/baseline.yaml",
    "configs/pooling_max.yaml",
    "configs/pooling_attention.yaml",
    "configs/cell_dim_128.yaml",
    "configs/cell_dim_512.yaml",
    "configs/grid_16x8.yaml",
    "configs/grid_64x8.yaml",
    "configs/grid_64x16.yaml",
    "configs/cnn_depth_4.yaml",
]
CONFIG_ARGS = " ".join(SWEEP_CONFIGS)
DRIVE_SWEEP_ROOT = "/content/drive/MyDrive/cnn_qwen_table_mcr/outputs"
!python scripts/run_sweep.py --configs {CONFIG_ARGS} --mirror-root {DRIVE_SWEEP_ROOT}